# Task 4: OpenRA RTS — PrometheusStar Curriculum Learning

## Evolutionary Agents for Real-Time Strategy Games

**Prometheus v0.97** | [Open in Colab](https://colab.research.google.com/github/pmcray/Prometheus_v0_PoC/blob/master/notebooks/task4_openra_demo.ipynb)

This notebook demonstrates the PrometheusStar evolutionary curriculum learning framework applied to **OpenRA** (open-source Command & Conquer engine).

No external OpenRA installation is required — a self-contained discrete-event simulator is built in.

### Architecture

```
  OpenRAEnvironment (Gym-compatible)
      ↕  20-D feature vector  ↕  15 discrete actions
  OpenRAStrategyAgent (5 evolved parameters)
      ↓ evolve via elitism + crossover + mutation
  OpenRABenchmark (N games, win_rate score)
      ↓ 4-stage curriculum
  Easy(70%) → Medium(55%) → Hard(40%) → Expert(25%)
```

### Curriculum Stages
| Stage | Opponent | Target Win Rate | AI Speed |
|---|---|---|---|
| 1 | Easy AI   | 70% | 0.5× |
| 2 | Medium AI | 55% | 1.0× |
| 3 | Hard AI   | 40% | 1.5× |
| 4 | Expert AI | 25% | 2.5× |

**Runtime**: ~3 minutes (CPU) for a quick 2-stage run

In [ ]:
# 1. Clone repo and install dependencies
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
    %cd Prometheus_v0_PoC
    !pip install -q numpy scipy matplotlib pytest
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

print('Setup complete')

## 1. Explore the OpenRA Environment

In [ ]:
from prometheus.openra_environment import (
    OpenRAEnvironment, OpenRAGameState, GameResult,
    FEATURE_SIZE, NUM_ACTIONS, ACTION_NAMES, DIFFICULTY_SPEED
)
import numpy as np
import matplotlib.pyplot as plt

print('OpenRA Environment')
print(f'  Feature size:  {FEATURE_SIZE}  (observation vector dimensions)')
print(f'  Action space:  {NUM_ACTIONS}  discrete actions')
print()
print('Actions:')
for i, name in enumerate(ACTION_NAMES):
    print(f'  {i:2d}. {name}')
print()
print('Difficulty speed multipliers:')
for diff, speed in DIFFICULTY_SPEED.items():
    bar = '█' * int(speed * 4)
    print(f'  {diff:<10} {speed}×  {bar}')

In [ ]:
# Explore the 20-dimensional game state vector
env = OpenRAEnvironment(difficulty='easy', seed=42)
state = env.reset()

print('Initial game state:')
print(f'  Credits:          {state.credits:.0f}')
print(f'  Income rate:      {state.income_rate:.2f}')
print(f'  Own units:        infantry={state.own_infantry}, vehicles={state.own_vehicles}')
print(f'  Enemy units:      infantry={state.enemy_infantry}, vehicles={state.enemy_vehicles}')
print(f'  Map control:      {state.map_control_pct:.1%}')
print(f'  Resource pct:     {state.resource_pct:.1%}')
print(f'  Game step:        {state.game_step}')
print(f'  Result:           {state.game_result}')

obs = env.get_observation()
print(f'\nObservation vector (normalised, shape={obs.shape}):')
feature_names = [
    'credits', 'income_rate', 'own_infantry', 'own_vehicles', 'own_aircraft',
    'own_buildings', 'own_harvesters', 'enemy_infantry', 'enemy_vehicles',
    'enemy_aircraft', 'enemy_buildings', 'map_control', 'resource_pct',
    'own_units_lost', 'enemy_units_lost', 'own_bld_lost', 'enemy_bld_lost',
    'game_step', 'own_strength', 'enemy_strength'
]
for i, (name, val) in enumerate(zip(feature_names, obs)):
    bar = '█' * int(abs(val) * 20)
    print(f'  [{i:2d}] {name:<20} {val:+6.3f}  {bar}')

## 2. Play a Sample Game Episode

In [ ]:
from prometheus.openra_agent import OpenRAStrategyAgent

# Create an agent and play one episode
agent = OpenRAStrategyAgent()
env   = OpenRAEnvironment(difficulty='easy', seed=123)

print('Agent strategy parameters:')
for k, v in agent.params.items():
    bar = '█' * int(v * 20)
    print(f'  {k:<20} {v:.3f}  {bar}')
print()

# Run one episode
state = env.reset()
agent.reset()

episode_rewards = []
own_strength    = []
enemy_strength  = []

done = False
max_steps = 500
step = 0

while not done and step < max_steps:
    action = agent.select_action(state)
    state, reward, done, info = env.step(action)
    episode_rewards.append(reward)
    own_strength.append(state.total_own_strength())
    enemy_strength.append(state.total_enemy_strength())
    step += 1

total_reward = sum(episode_rewards)
print(f'Episode complete — {step} steps')
print(f'  Total reward:      {total_reward:+.1f}')
print(f'  Result:            {state.game_result.value}')
print(f'  Own units lost:    {state.own_units_lost}')
print(f'  Enemy units lost:  {state.enemy_units_lost}')
print(f'  Own bldgs lost:    {state.own_buildings_lost}')
print(f'  Enemy bldgs lost:  {state.enemy_buildings_lost}')

# Plot strength over time
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.plot(own_strength,   color='royalblue',  linewidth=1.5, label='Own strength')
ax.plot(enemy_strength, color='firebrick',  linewidth=1.5, label='Enemy strength')
ax.fill_between(range(len(own_strength)),   own_strength,   alpha=0.2, color='royalblue')
ax.fill_between(range(len(enemy_strength)), enemy_strength, alpha=0.2, color='firebrick')
ax.set_xlabel('Game Step')
ax.set_ylabel('Total Strength')
ax.set_title(f'Military Strength over Time\n({state.game_result.value})',
             fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
cumulative = np.cumsum(episode_rewards)
ax.plot(cumulative, color='darkorange', linewidth=1.5)
ax.fill_between(range(len(cumulative)), cumulative, alpha=0.2, color='darkorange')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Game Step')
ax.set_ylabel('Cumulative Reward')
ax.set_title('Cumulative Reward over Episode', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Single Episode — Easy AI', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Evolutionary Operators — Mutation & Crossover

In [ ]:
np.random.seed(0)

# Show mutation in action
parent = OpenRAStrategyAgent()
print('Parent strategy:')
for k, v in parent.params.items():
    print(f'  {k:<20} {v:.4f}')

print('\nMutated children (mutation_rate=0.25):')
print(f'  {"Param":<20}', end='')
for i in range(4):
    print(f'  Child {i+1}  ', end='')
print()
for k in parent.params:
    print(f'  {k:<20}', end='')
    for i in range(4):
        np.random.seed(i * 10 + 1)
        child = parent.mutate(mutation_rate=0.25)
        delta = child.params[k] - parent.params[k]
        sign  = '+' if delta >= 0 else ''
        print(f'  {child.params[k]:.4f}({sign}{delta:.3f})', end='')
    print()

# Show crossover
np.random.seed(42)
parent_a = OpenRAStrategyAgent({'aggression':0.8,'economy_focus':0.2,'expansion_rate':0.7,'tech_priority':0.9,'defense_bias':0.1})
parent_b = OpenRAStrategyAgent({'aggression':0.2,'economy_focus':0.8,'expansion_rate':0.3,'tech_priority':0.1,'defense_bias':0.9})
child_ab  = parent_a.crossover(parent_b)

print('\nCrossover (uniform, p=0.5 per parameter):')
print(f'  {"Param":<20} {"Parent A":>10} {"Parent B":>10} {"Child":>10} {"From":>8}')
print('  ' + '-' * 52)
for k in parent_a.params:
    va = parent_a.params[k]; vb = parent_b.params[k]; vc = child_ab.params[k]
    src = 'A' if abs(vc - va) < 1e-9 else 'B'
    print(f'  {k:<20} {va:>10.4f} {vb:>10.4f} {vc:>10.4f} {src:>8}')

## 4. Curriculum Overview

In [ ]:
from benchmarks.openra_benchmark import OPENRA_CURRICULUM, estimate_openra_training_time

print('=== OpenRA Curriculum ===')
print()
for i, cfg in enumerate(OPENRA_CURRICULUM):
    print(f'Stage {i+1}: {cfg.name}')
    print(f'  Difficulty:   {cfg.difficulty}')
    print(f'  Skill level:  {cfg.skill_level}/10')
    print(f'  Target:       {cfg.target_win_rate:.0%} win rate')
    print(f'  Description:  {cfg.description}')
    print()

# Time estimates
print('Training time estimates (per stage):')
configs = [
    (6,  10, 4, 'Quick demo (this notebook)'),
    (8,  15, 4, 'Standard run'),
    (12, 25, 5, 'Extended run'),
]
print(f'  {"Config":<30} {"Games":>8} {"Minutes":>10}')
print('  ' + '-' * 52)
for pop, gens, gpe, desc in configs:
    est = estimate_openra_training_time(pop, gens, gpe)
    print(f'  {desc:<30} {est["total_games"]:>8} {est["total_minutes"]:>10.0f} min')

## 5. Run Stage 1 — Basic Controls (Easy AI)

This runs the full evolutionary loop for Stage 1 (Easy AI, target 70% win rate).

In [ ]:
from run_prometheusstar_openra import run_stage

# Quick run: smaller population and fewer generations for fast demo
stage1_cfg = {
    'stage':         1,
    'name':          'Basic Controls',
    'difficulty':    'easy',
    'population':    6,
    'generations':   10,
    'games_per_eval': 4,
    'mutation_rate': 0.25,
    'target':        0.70,
}

result1 = run_stage(stage1_cfg, seed=42, verbose=True)
print(f'\nStage 1 complete: best_win_rate={result1["best_win_rate"]:.1%}')

## 6. Run Stage 2 — Resource Management (Medium AI)

In [ ]:
stage2_cfg = {
    'stage':         2,
    'name':          'Resource Management',
    'difficulty':    'medium',
    'population':    6,
    'generations':   10,
    'games_per_eval': 4,
    'mutation_rate': 0.20,
    'target':        0.55,
}

result2 = run_stage(stage2_cfg, seed=42, verbose=True)
print(f'\nStage 2 complete: best_win_rate={result2["best_win_rate"]:.1%}')

## 7. Learning Curve Visualisation

In [ ]:
from run_prometheusstar_openra import plot_learning_curves, print_summary

stage_results = [r for r in [result1, result2] if r is not None]
print_summary(stage_results)

# Manual learning curve plot (so it shows inline in the notebook)
fig, ax = plt.subplots(figsize=(11, 5))
colours  = ['royalblue', 'darkorange', 'green', 'red']
gen_offset = 0

for i, result in enumerate(stage_results):
    log    = result['generation_log']
    gens   = [gen_offset + l['generation'] for l in log]
    best   = [l['best_win_rate'] for l in log]
    mean   = [l['mean_win_rate'] for l in log]
    target = result['target']
    colour = colours[i % len(colours)]

    ax.plot(gens, [b*100 for b in best], color=colour, linewidth=2.5,
            label=f"Stage {result['stage']}: {result['name']} (best)")
    ax.plot(gens, [m*100 for m in mean], color=colour, linewidth=1.2,
            linestyle='--', alpha=0.6, label='mean')
    ax.axhline(target*100, color=colour, linewidth=1.2, linestyle=':',
               alpha=0.9, label=f'Target {target:.0%}')
    gen_offset += len(log)

ax.set_xlabel('Generation (cumulative)', fontsize=11)
ax.set_ylabel('Win Rate (%)', fontsize=11)
ax.set_title('PrometheusStar OpenRA — Curriculum Learning Curves',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.set_ylim(-5, 105)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Best Agent Analysis

In [ ]:
# Inspect the best evolved agent
best_result = max(stage_results, key=lambda r: r['best_win_rate'])
best_agent  = best_result['best_agent']

print(f'Best agent from Stage {best_result["stage"]} ({best_result["name"]})')
print(f'  Best win rate:  {best_result["best_win_rate"]:.1%}')
print(f'  Target:         {best_result["target"]:.0%}')
print(f'  Status:         {"PASSED" if best_result["target_met"] else "FAILED"}')
print()
print('Strategy parameters:')

params = best_agent.params
param_descriptions = {
    'aggression':      'How often to attack vs. defend',
    'economy_focus':   'Investment in harvesters and income',
    'expansion_rate':  'Rate of base/territory expansion',
    'tech_priority':   'Investment in advanced units (vehicles, aircraft)',
    'defense_bias':    'Preference for defensive positioning',
}
for k, v in params.items():
    bar = '█' * int(v * 20)
    desc = param_descriptions.get(k, '')
    print(f'  {k:<20} {v:.4f}  {bar}  ← {desc}')

# Radar chart of agent parameters
import matplotlib.pyplot as plt
import numpy as np

labels = list(params.keys())
values = list(params.values())
values += values[:1]  # Close the polygon
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(subplot_kw={'polar': True}, figsize=(6, 6))
ax.plot(angles, values, color='royalblue', linewidth=2)
ax.fill(angles, values, color='royalblue', alpha=0.25)
ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
ax.set_title(f'Best Agent Strategy Profile\n(Stage {best_result["stage"]}, win={best_result["best_win_rate"]:.1%})',
             fontsize=12, fontweight='bold', pad=15)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 9. Baseline Agent Comparison

In [ ]:
from benchmarks.openra_benchmark import (
    OpenRABenchmark, RandomOpenRAAgent, RushOpenRAAgent, EconomyOpenRAAgent
)

benchmark = OpenRABenchmark(difficulty='easy', seed=999)

agents = [
    ('Random',   RandomOpenRAAgent()),
    ('Rush',     RushOpenRAAgent()),
    ('Economy',  EconomyOpenRAAgent()),
    ('Evolved',  best_agent),
]

print('=== Baseline Agent Comparison (Easy AI, 5 games) ===')
print(f'  {"Agent":<15} {"Win Rate":>10} {"Avg Reward":>12} {"Avg Steps":>12} {"Wins/Loss/Draw":>15}')
print('  ' + '-' * 65)

results = {}
for name, agent_obj in agents:
    r = benchmark.evaluate_agent(agent_obj, num_games=5)
    results[name] = r
    print(f'  {name:<15} {r["win_rate"]:>10.1%} {r["avg_reward"]:>+12.1f} '
          f'{r["avg_game_length"]:>12.0f} '
          f'  {r["wins"]}/{r["losses"]}/{r["draws"]}')

# Bar chart
names   = list(results.keys())
wr      = [results[n]['win_rate'] * 100 for n in names]
colors  = ['#7f8c8d', '#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(names, wr, color=colors, edgecolor='black', linewidth=1.2)
ax.axhline(70, color='black', linestyle='--', linewidth=1.5,
           label='Stage 1 target (70%)')
for bar, val in zip(bars, wr):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1,
            f'{val:.0f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Win Rate (%)', fontsize=11)
ax.set_title('Agent Comparison — Easy AI (5 games each)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Full 4-Stage Curriculum (Optional — ~8 minutes)

Uncomment the cell below to run all 4 curriculum stages.

In [ ]:
# OPTIONAL — uncomment to run the full 4-stage curriculum

# from run_prometheusstar_openra import main
# all_results = main(
#     max_stages=4,
#     population=8,
#     generations=12,
#     games_per_eval=4,
#     seed=42,
#     plot=True,
#     verbose=True,
# )

print('Full curriculum run is commented out by default.')
print('Uncomment the lines above to run all 4 stages (~8 minutes).')

## Summary

| Component | Description |
|---|---|
| `OpenRAEnvironment` | Gym-compatible simulator (no real OpenRA needed) |
| `OpenRAGameState` | 20-D feature vector (credits, unit counts, map control, etc.) |
| `OpenRAStrategyAgent` | 5-parameter evolved agent (aggression, economy, expansion, tech, defense) |
| `mutate()` | Gaussian perturbation, clipped to [0,1] |
| `crossover()` | Uniform crossover (p=0.5 per parameter) |
| `OpenRABenchmark` | N-game evaluation returning win_rate + game summaries |
| Curriculum | 4 stages: easy(70%)→medium(55%)→hard(40%)→expert(25%) |
| Elitism | Top 30% agents carried over unchanged each generation |

**Key insight**: No hand-crafted strategy is needed. The evolutionary loop discovers that high-economy agents beat easy AI while balanced aggression is needed for expert AI — purely through self-play fitness signals.

**Extension**: Replace `_SimulatedOpenRA` with the `openra-python` bridge package to train against the real OpenRA engine with no code changes.